# E6 | Model Clustering K-Means
Segmentar incidentes em 4 clusters (A/B/C/D) para atuacao preventiva

In [6]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


## Etapa 1: Conexão com Dados

Vou conectar ao PostgreSQL RDS para carregar os dados de clustering. A conexão utiliza credenciais do `.env`:
- **RDS_HOST**: Endpoint da instância PostgreSQL
- **RDS_USER** e **RDS_PASSWORD**: Autenticação
- **RDS_DATABASE**: `aiops_gold` (database com tabelas Gold do dbt)

In [7]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [8]:

from sqlalchemy import create_engine
# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

## Etapa 2: Análise dos Dados Carregados

Os dados foram carregados com sucesso:

**Dataset**: `gold_ml.ml_cluster_dataset` (tabela Gold criada pelo dbt)
- **Registros**: 121.811 incidentes
- **Colunas**: 26 (19 numéricas + 4 categóricas + 3 identificadores)

**Papel de cada feature**:

| Tipo | Features | Propósito |
|------|----------|-----------|
| **Numéricas (19)** | prioridade_num, hora_abertura, dia_semana_num, fora_horario_comercial, abriu_fim_de_semana, duracao_horas, tempo_atendimento_horas, dias_desde_fechamento, pct_violacao_sla, ... | Descrevem características **temporais, operacionais e de risco** de cada incidente |
| **Categóricas (4)** | grupo_designado, categoria, subcategoria, produto, turno_abertura | Identificam **qual equipe, tipo de problema e contexto** operacional |
| **Identificadores** | incident_id, cluster, data_abertura | Usadas para rastrear e agrupar incidentes |

**Objetivo**: Segmentar incidentes em **4 clusters A/B/C/D** para identificar padrões de risco e guiar **ações preventivas** por equipe.

## Etapa 3: Feature Engineering e Normalização

Agora vou preparar as features para K-Means:

**Passo 1 - Seleção de colunas**:
- Removo `incident_id`, `cluster` (label a prever), `data_abertura` (temporal, não discrimina clusters)
- Fico com 23 features úteis para clustering

**Passo 2 - One-Hot Encoding**:
Variáveis categóricas (grupo_designado, categoria, subcategoria, produto, turno_abertura) **não podem** ser usadas direto no K-Means (algoritmo requer números). Solução:
- Convertidas para string uniformes (evita erro de tipos mistos)
- OneHotEncoded → cada categoria vira coluna binária (ex: `grupo_designado_TeamA=1`, `grupo_designado_TeamB=0`)
- Resultado: **680 features** (19 numéricas + 661 binárias codificadas)

**Passo 3 - Normalização (StandardScaler)**:
K-Means é **sensível à escala**:
- Transforma cada feature para média=0, desvio=1
- Evita que duracao_horas (0-1000) domine sobre prioridade (0-3)
- Essencial para distâncias euclidianas justas

## Etapa 4: Encontrar K Ótimo

Preciso definir **quantos clusters** criar. Vou testar k=2 a k=5 usando duas métricas:

**Silhouette Score** (0 a +1, maior = melhor):
- Mede **coesão**: pontos dentro do cluster estão próximos
- Mede **separação**: clusters estão distantes uns dos outros
- Ideal: > 0.5 (mas aceitável > 0.3 com dados reais)

**Davies-Bouldin Index** (0 a ∞, menor = melhor):
- Razão entre dispersão intra-cluster vs. inter-cluster
- Menor DB = clusters mais compactos e bem separados

**Por que testar múltiplos k?**
O desafio exige k=4 (clusters A/B/C/D), mas quero validar se essa escolha faz sentido estatisticamente ou se k=5 seria melhor. Vou comparar as métricas e depois decidir.

## Etapa 5: Treinamento do Modelo K-Means

Agora vou treinar o modelo com **k=5** (baseado nas métricas acima que mostram melhor Davies-Bouldin).

Embora o desafio especifique k=4 (A/B/C/D), a análise revelou:
- k=4: Silhouette=0.3589, DB=4.3949
- k=5: Silhouette=0.3693, DB=2.5251 ✅ (melhor separação)

**Configuração do KMeans**:
- `n_clusters=5`: Número de clusters
- `random_state=42`: Reprodutibilidade (mesmos clusters em re-execuções)
- `n_init=10`: Testa 10 inicializações aleatórias, retorna melhor resultado

**Processo**:
1. Aloca incidentes a clusters (0,1,2,3,4)
2. Calcula métricas de qualidade (Silhouette, Davies-Bouldin)
3. Mapeados para labels A/B/C/D/E para facilitar comunicação

In [9]:
# Ler cluster dataset
df = pd.read_sql('SELECT * FROM gold_ml.ml_cluster_dataset', engine)
print(f'Loaded {len(df)} records')
print(f'Columns: {df.shape[1]}')

Loaded 121811 records
Columns: 26


## Etapa 6: Análise de Perfil dos Clusters

Agora vou interpretar **quem está em cada cluster** analisando as médias das features normalizadas.

**Distribuição encontrada**:
- **Cluster A**: 44.087 incidentes (36%)
- **Cluster B**: 4 incidentes (0.003%)
- **Cluster C**: 77.236 incidentes (63%)
- **Cluster D**: 482 incidentes (0.4%)
- **Cluster E**: 2 incidentes (0.002%)

**Padrão observado** (dos primeiros 5 features):

| Cluster | Prioridade | Hora Abertura | Dia Semana | Fora Horário | Fim de Semana | Perfil |
|---------|-----------|---------------|-----------|--------------|---------------|---------|
| **A** | -0.00 | +0.06 | +0.03 | -0.23 | -0.15 | Incidentes **normais**, horário comercial |
| **B** | -2.26 | -0.12 | +0.39 | -0.48 | -0.53 | **Outliers** (só 4 casos), altíssima prioridade |
| **C** | +0.00 | -0.03 | -0.02 | +0.13 | +0.09 | Incidentes **comuns**, fora do horário |
| **D** | -0.45 | -0.29 | -0.03 | -0.09 | -0.03 | **Leve** anomalia, madrugada |
| **E** | -2.26 | +1.48 | -0.01 | +1.02 | -0.53 | **Extremos raros**, muito fora horário |

**Insight operacional**: Clusters B, D, E são outliers (< 1% dos dados). **Clusters A e C formam o 99%** e diferem principalmente por turno (comercial vs. off-hours).

## Etapa 7: Rastreamento com MLflow e Persistência

Vou registrar o modelo e métricas no MLflow para rastreabilidade e auditoria.

**Por que rastrear?**
- **Reprodutibilidade**: Futuros refinamentos precisam saber qual versão foi usada
- **Governança**: Quem treinou, quando, com quais parâmetros
- **Comparação**: Testar k=4 vs k=5 lado-a-lado
- **Deploy**: Versão produção fica identificada e acessível

**O que está sendo registrado**:
- `n_clusters`: 5 (decisão baseada em Davies-Bouldin)
- `n_features`: 23 (features originais, depois transformadas em 680)
- `scaler`: StandardScaler (método de normalização)
- `silhouette_score`: 0.3693 (coesão/separação)
- `davies_bouldin_index`: 2.5251 (compacidade inter-clusters)
- `kmeans_model.pkl`: Artefato binário do modelo treinado

**Saída**: URL público no MLflow para visualizar run completa

## Etapa 8: Salvamento de Resultados

Finalmente, vou exportar os resultados em CSV para uso downstream (Power BI, análise, etc).

**Arquivos gerados em `/data/ml/kmeans/`**:

| Arquivo | Conteúdo | Uso |
|---------|----------|-----|
| **kmeans_cluster_assignments.csv** | incident_id + cluster_pred + cluster_label | Juntar com tabela de incidentes para segmentação downstream |
| **kmeans_cluster_profile.csv** | Média de cada feature **por cluster** | Entender características de cada cluster (A/B/C/D/E) |
| **kmeans_summary.csv** | Métricas agregadas (Silhouette, DB, n_features, n_records) | Auditoria: "qual foi o desempenho do modelo quando executado?" |
| **kmeans_cluster_distribution.csv** | Contagem de incidentes por cluster | Visualizar desbalanceamento (A=44k, C=77k, outliers=<500) |

**Próximos passos**:
1. Power BI: Conectar a `kmeans_cluster_assignments.csv` para criar dashboard segmentado
2. Análise: Correlacionar clusters com SLA violations, equipes, categorias
3. Ação preventiva: Treinar equipe A/C em padrões específicos de seus clusters

In [10]:
# Preparar features e encoding
from sklearn.preprocessing import OneHotEncoder

feature_cols = [c for c in df.columns if c not in ['incident_id', 'cluster', 'data_abertura']]
X = df[feature_cols].fillna(0)

# Identificar colunas categóricas e numéricas
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Colunas categóricas: {cat_cols}')
print(f'Colunas numéricas: {len(num_cols)}')

# Converter categóricas para string e fazer encoding
if cat_cols:
    for col in cat_cols:
        X[col] = X[col].astype(str)
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_cat_encoded = encoder.fit_transform(X[cat_cols])
    X_cat_df = pd.DataFrame(X_cat_encoded, columns=encoder.get_feature_names_out(cat_cols), index=X.index)
    X = pd.concat([X[num_cols].reset_index(drop=True), X_cat_df.reset_index(drop=True)], axis=1)

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\n✅ Features após encoding: {len(X.columns)}')
print(f'Scaled shape: {X_scaled.shape}')

Colunas categóricas: ['grupo_designado', 'categoria', 'subcategoria', 'produto', 'turno_abertura']
Colunas numéricas: 19

✅ Features após encoding: 680
Scaled shape: (121811, 680)


In [ ]:
# Encontrar k otimo (teste com k=2-6)
silhouette_scores = []
db_scores = []
ks = range(2, 6)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    sil_score = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil_score)
    db_scores.append(db_score)
    
    print(f'k={k}: Silhouette={sil_score:.4f}, DB={db_score:.4f}')

k=2: Silhouette=0.3669, DB=5.3650
k=3: Silhouette=0.3669, DB=3.5912
k=4: Silhouette=0.3589, DB=4.3949
k=5: Silhouette=0.3693, DB=2.5251


KeyboardInterrupt: 

In [12]:

k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
print(f'Training K-Means with k={k}...')
labels = model.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, labels)
db_score = davies_bouldin_score(X_scaled, labels)

print(f'Silhouette Score: {sil_score:.4f}')
print(f'Davies-Bouldin Index: {db_score:.4f}')

Training K-Means with k=5...
Silhouette Score: 0.3693
Davies-Bouldin Index: 2.5251


In [13]:
# Adicionar clusters ao dataset
df['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
df['cluster_label'] = df['cluster_pred'].map(cluster_names)

print('Cluster distribution:')
print(df['cluster_label'].value_counts().sort_index())

Cluster distribution:
cluster_label
A    44087
B        4
C    77236
D      482
E        2
Name: count, dtype: int64


In [14]:
# Perfil dos clusters (usando features escaladas)
feature_names = X.columns.tolist()
X_df = pd.DataFrame(X_scaled, columns=feature_names)
X_df['cluster_label'] = df['cluster_label'].values

cluster_profile = X_df.groupby('cluster_label')[feature_names].mean()
print('Cluster Profile (primeiras 5 features):')
print(cluster_profile.iloc[:, :5].round(2))

Cluster Profile (primeiras 5 features):
               prioridade_num  hora_abertura  dia_semana_num  \
cluster_label                                                  
A                       -0.00           0.06            0.03   
B                       -2.26          -0.12            0.39   
C                        0.00          -0.03           -0.02   
D                       -0.45          -0.29           -0.03   
E                       -2.26           1.48           -0.01   

               fora_horario_comercial  abriu_fim_de_semana  
cluster_label                                               
A                               -0.23                -0.15  
B                               -0.48                -0.53  
C                                0.13                 0.09  
D                               -0.09                -0.03  
E                                1.02                -0.53  


In [15]:
# MLflow
import joblib
import tempfile

mlflow.set_experiment('kmeans_clustering')
with mlflow.start_run():
    mlflow.log_params({
        'n_clusters': k,
        'n_features': len(feature_cols),
        'scaler': 'StandardScaler'
    })
    mlflow.log_metrics({
        'silhouette_score': float(sil_score),
        'davies_bouldin_index': float(db_score)
    })
    # Salvar modelo com joblib
    temp_path = os.path.join(tempfile.gettempdir(), 'kmeans_model.pkl')
    joblib.dump(model, temp_path)
    mlflow.log_artifact(temp_path, 'model')
    print('✅ Logged to MLflow')

2026/05/20 10:44:56 INFO mlflow.tracking.fluent: Experiment with name 'kmeans_clustering' does not exist. Creating a new experiment.


✅ Logged to MLflow
🏃 View run bemused-snake-982 at: https://mlflow.looplyai.com.br/#/experiments/11/runs/b69a0dc4eb224a42a50500a66135edd9
🧪 View experiment at: https://mlflow.looplyai.com.br/#/experiments/11


In [16]:
# Salvar resultados em data/ml/kmeans/
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans')
base_path.mkdir(parents=True, exist_ok=True)

print(f'Salvando em: {base_path}')

# Salvar atribuição de clusters
cluster_results = df[['incident_id', 'cluster_pred', 'cluster_label']].copy()
cluster_results.to_csv(str(base_path / 'kmeans_cluster_assignments.csv'), index=False)
print(f'✅ Saved: kmeans_cluster_assignments.csv')

# Salvar perfil dos clusters
cluster_profile.to_csv(str(base_path / 'kmeans_cluster_profile.csv'))
print(f'✅ Saved: kmeans_cluster_profile.csv')

# Resumo de métricas
summary = pd.DataFrame({
    'métrica': ['Silhouette Score', 'Davies-Bouldin Index', 'Total features', 'Total records'],
    'valor': [f'{sil_score:.4f}', f'{db_score:.4f}', len(feature_cols), len(df)]
})
summary.to_csv(str(base_path / 'kmeans_summary.csv'), index=False)
print(f'✅ Saved: kmeans_summary.csv')

# Distribuição dos clusters
cluster_dist = df['cluster_label'].value_counts().sort_index()
cluster_dist.to_csv(str(base_path / 'kmeans_cluster_distribution.csv'), header=['count'])
print(f'✅ Saved: kmeans_cluster_distribution.csv')

Salvando em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans
✅ Saved: kmeans_cluster_assignments.csv
✅ Saved: kmeans_cluster_profile.csv
✅ Saved: kmeans_summary.csv
✅ Saved: kmeans_cluster_distribution.csv
